# Preprocessing

cleaning up the raw image so we can work with it later. basically going from a messy phone photo to a clean binary (black/white) image where the handwriting is visible.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def show(images, titles, figsize=(16, 5)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, t in zip(axes, images, titles):
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(t)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## load the image

In [ ]:
img_path = '../data/raw_samples/eq1.png'
original = cv2.imread(img_path)
if original is None:
    raise FileNotFoundError(f'cant find {img_path}')

print(f'shape: {original.shape}')
show([original], ['original'])

## grayscale

dont need color info, just need to know dark (ink) vs light (paper)

In [ ]:
gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
print(f'before: {original.shape}, after: {gray.shape}')
show([original, gray], ['color', 'grayscale'])

## gaussian blur

phone camera adds noise. gaussian blur smooths that out. tried different kernel sizes to see what works best

In [ ]:
blur3 = cv2.GaussianBlur(gray, (3, 3), 0)
blur5 = cv2.GaussianBlur(gray, (5, 5), 0)
blur9 = cv2.GaussianBlur(gray, (9, 9), 0)

show([gray, blur3, blur5, blur9],
     ['no blur', '3x3', '5x5', '9x9'])

# 5x5 seemed like the sweet spot - 3x3 barely does anything, 9x9 blurs too much
blurred = blur5

## contrast enhancement (CLAHE)

sometimes the lighting is uneven (one side of the paper is brighter). tried global histogram equalization first but it messed up some areas. CLAHE does it in small tiles so it handles uneven lighting better

In [ ]:
# tried global first
global_eq = cv2.equalizeHist(blurred)

# then CLAHE - this one works way better
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_eq = clahe.apply(blurred)

show([blurred, global_eq, clahe_eq],
     ['blurred', 'global eq', 'CLAHE'])

# histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, img, t in zip(axes, [blurred, global_eq, clahe_eq], ['before', 'global', 'CLAHE']):
    ax.hist(img.ravel(), 256, (0, 255), color='steelblue', alpha=0.7)
    ax.set_title(t)
    ax.set_xlim(0, 255)
plt.tight_layout()
plt.show()

enhanced = clahe_eq

## thresholding

need to make it binary - ink=white, paper=black. tried 3 methods

In [ ]:
# otsu - picks one global threshold automatically
_, otsu = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print(f'otsu picked T={_:.0f}')

# adaptive mean
adapt_mean = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV, 15, 10)

# adaptive gaussian - best one for this
adapt_gauss = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY_INV, 15, 10)

show([enhanced, otsu, adapt_mean, adapt_gauss],
     ['input', 'otsu', 'adaptive mean', 'adaptive gaussian'])

# adaptive gaussian handles uneven lighting best
binary = adapt_gauss

## morphological cleanup

thresholding leaves some noise dots and broken strokes. opening removes the dots, closing fills the gaps

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

# opening = erode then dilate (kills small dots)
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

# closing = dilate then erode (fills small gaps)
cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=1)

show([binary, opened, cleaned],
     ['after threshold', 'after opening', 'after closing'])

## all steps together

In [ ]:
show([original, gray, blurred, enhanced, binary, cleaned],
     ['original', 'grayscale', 'blurred', 'CLAHE', 'threshold', 'cleaned'],
     figsize=(20, 4))

print(f'input:  {original.shape}')
print(f'output: {cleaned.shape}')

## wrapping it as a function

putting the whole pipeline into one function so i can just call it from the next notebooks without copy pasting everything

In [ ]:
def preprocess(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'cant load {image_path}')
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)
    
    binary = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 15, 10)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel, iterations=1)
    
    return cleaned

# quick test
result = preprocess('../data/raw_samples/eq1.png')
show([cv2.imread('../data/raw_samples/eq1.png'), result], ['input', 'output'])

In [ ]:
# save the result
os.makedirs('../data/processed', exist_ok=True)
cv2.imwrite('../data/processed/eq1_binary.png', result)
print('saved')